In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
%pip install -q optuna scikit-learn pandas pyarrow mlflow

In [22]:
import sys
sys.path.append('/content/drive/MyDrive/repositories/aq_edge')

In [23]:
%cd /content/drive/MyDrive/repositories/aq_edge/colab
%run mlflow_colab.py

/content/drive/MyDrive/repositories/aq_edge/colab
MLflow server ready: http://127.0.0.1:5000


In [24]:
%cd /content/drive/MyDrive/repositories/aq_edge

import os
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"

# Ausführen des Skripts mit expliziter Übergabe der Parameter für die Station USAM und 2 Trials
!python hyperparameter_optuna.py --data.station 'USAM' --optuna.n_trials 100

/content/drive/MyDrive/repositories/aq_edge/colab
[2026-10-22 08:24:18] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/preprocessing_20261022_082418.log
[2026-10-22 08:24:18] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/metrics_20261022_082418.log
[2026-10-22 08:24:20] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/main_20261022_082420.log
INFO:__main__:Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/main_20261022_082420.log
[2026-10-22 08:24:20] INFO: Starting CO2 forecasting hyperparameter optimization
INFO:__main__:Starting CO2 forecasting hyperparameter optimization
[2026-10-22 08:24:20] INFO: Device: cuda
INFO:__main__:Device: cuda
[2026-10-22 08:24:21] INFO: Station: USAM
INFO:__main__:Station: USAM
[2026-10-22 08:24:21] INFO: Features: ['TEM', 'HUM', 'VOC']
INFO:__main__:Features: ['TEM', 'HUM', 'VOC']
[2026-10-22 08:24:21] INFO: Target: CO2
INFO:__main__:Target: CO

In [25]:
import os
import shutil
import sqlite3

os.makedirs(BACKUP, exist_ok=True)

# Consistent SQLite backup
src = sqlite3.connect(DB)
dst = sqlite3.connect("/content/mlflow_snapshot.db")
with dst:
    src.backup(dst)
src.close()
dst.close()
shutil.copy2("/content/mlflow_snapshot.db", f"{BACKUP}/mlflow.db")

# Preserve experiment artifacts
if os.path.isdir(ARTIFACTS):
    shutil.copytree(
        ARTIFACTS, f"{BACKUP}/mlartifacts", dirs_exist_ok=True
    )

print("MLflow database and artifacts backed up to Drive.")

MLflow database and artifacts backed up to Drive.


In [26]:
from pathlib import Path
import shutil

base_repo_dir = Path("/content/drive/MyDrive/repositories/aq_edge")
station_name = 'USAM'

# Recursively find any generated output files under the repository
json_sources = list(base_repo_dir.rglob(f"best_co2_{station_name}*.json"))
csv_sources = list(base_repo_dir.rglob(f"trials_co2_{station_name}*.csv"))

target_station_dir = base_repo_dir / "optuna_results" / station_name
target_station_dir.mkdir(parents=True, exist_ok=True)

print(f"Found JSON files at: {[str(p) for p in json_sources]}")
print(f"Found CSV files at: {[str(p) for p in csv_sources]}")

# Move found files directly to the target folder, avoiding moving a file onto itself
for src in json_sources:
    dest_path = target_station_dir / src.name
    if src.resolve() != dest_path.resolve():
        shutil.move(str(src), str(dest_path))
        print(f"Moved: {src.resolve()} -> {dest_path.resolve()}")

for src in csv_sources:
    dest_path = target_station_dir / src.name
    if src.resolve() != dest_path.resolve():
        shutil.move(str(src), str(dest_path))
        print(f"Moved: {src.resolve()} -> {dest_path.resolve()}")

Found JSON files at: ['/content/drive/MyDrive/repositories/aq_edge/optuna_results/best_co2_USAM_base_lstm_test.json', '/content/drive/MyDrive/repositories/aq_edge/mlflow_backup/mlartifacts/2/159087acb4cb4530bb4d82a03ac99996/artifacts/optuna/best_co2_USAM_base_lstm_test.json', '/content/drive/MyDrive/repositories/aq_edge/mlflow_backup/mlartifacts/2/9e14647d3e5b4d8783b95f408a848c50/artifacts/optuna/best_co2_USAM_base_lstm_test.json', '/content/drive/MyDrive/repositories/aq_edge/mlflow_backup/mlartifacts/2/d101f33e3c22445fb0c839fd5b0c8fdf/artifacts/optuna/best_co2_USAM_base_lstm_test.json']
Found CSV files at: ['/content/drive/MyDrive/repositories/aq_edge/optuna_results/trials_co2_USAM_base_lstm_test.csv', '/content/drive/MyDrive/repositories/aq_edge/mlflow_backup/mlartifacts/2/159087acb4cb4530bb4d82a03ac99996/artifacts/optuna/trials_co2_USAM_base_lstm_test.csv', '/content/drive/MyDrive/repositories/aq_edge/mlflow_backup/mlartifacts/2/9e14647d3e5b4d8783b95f408a848c50/artifacts/optuna/tria